In [1]:
import torch

torch.manual_seed(42)

x = torch.randn(500,2)
y = ((x[:,0]**2) + (x[:,1]**2) > 1.0 ).float()

In [2]:
x.shape

torch.Size([500, 2])

In [3]:
y.shape

torch.Size([500])

In [4]:
from torch.utils.data import TensorDataset, DataLoader

train_size = 400

x_train = x[:train_size]
y_train = y[:train_size]

x_test = x[train_size:]
y_test = y[train_size:]

In [5]:
train_dataset = TensorDataset(
    x_train,
    y_train
)

test_dataset = TensorDataset(
    x_test,
    y_test
)

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size= 32,
    shuffle= True
)

In [7]:
import torch.nn as nn

class BinaryANN(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2,16),
            nn.ReLU(),
            nn.Linear(16,1)
        )

    def forward(self,x):
        return self.network(x)

In [8]:
model = BinaryANN()

In [9]:
loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [10]:
num_epochs = 100

for epoch in range(num_epochs):

    model.train()

    for x_batch,y_batch in train_loader:

        logits = model(x_batch)

        loss = loss_fn(
            logits.squeeze(),
            y_batch
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}, "
                f"Loss: {loss.item():.4f}"
            )

Epoch 10, Loss: 0.5870
Epoch 10, Loss: 0.6593
Epoch 10, Loss: 0.6116
Epoch 10, Loss: 0.6039
Epoch 10, Loss: 0.6142
Epoch 10, Loss: 0.5744
Epoch 10, Loss: 0.6109
Epoch 10, Loss: 0.6249
Epoch 10, Loss: 0.5945
Epoch 10, Loss: 0.6101
Epoch 10, Loss: 0.6161
Epoch 10, Loss: 0.6080
Epoch 10, Loss: 0.6131
Epoch 20, Loss: 0.5305
Epoch 20, Loss: 0.5910
Epoch 20, Loss: 0.5278
Epoch 20, Loss: 0.5467
Epoch 20, Loss: 0.5259
Epoch 20, Loss: 0.5844
Epoch 20, Loss: 0.6140
Epoch 20, Loss: 0.4882
Epoch 20, Loss: 0.5721
Epoch 20, Loss: 0.5333
Epoch 20, Loss: 0.5390
Epoch 20, Loss: 0.5573
Epoch 20, Loss: 0.5663
Epoch 30, Loss: 0.4528
Epoch 30, Loss: 0.4968
Epoch 30, Loss: 0.4786
Epoch 30, Loss: 0.4917
Epoch 30, Loss: 0.5449
Epoch 30, Loss: 0.5226
Epoch 30, Loss: 0.4699
Epoch 30, Loss: 0.4879
Epoch 30, Loss: 0.5042
Epoch 30, Loss: 0.5117
Epoch 30, Loss: 0.5064
Epoch 30, Loss: 0.4793
Epoch 30, Loss: 0.4274
Epoch 40, Loss: 0.4149
Epoch 40, Loss: 0.4414
Epoch 40, Loss: 0.3976
Epoch 40, Loss: 0.4609
Epoch 40, L

In [11]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        logits = model(X_batch)

        probabilities = torch.sigmoid(
            logits.squeeze()
        )

        predictions = (
            probabilities >= 0.5
        ).float()

        correct += (
            predictions == y_batch
        ).sum().item()

        total += y_batch.size(0)

accuracy = correct / total

print("Accuracy:", accuracy)

Accuracy: 0.96


# 📚 Module 15.3 — Binary Classification ANN — Complete Notes

## 1. Core Concept 🔥🔥🔥

A **Binary Classification ANN** predicts one of two classes:

```text
Class 0
Class 1
```

Unlike the classical perceptron, we use a **multi-layer neural network with nonlinear activation**.

### Architecture

```text
Input
  ↓
Linear
  ↓
ReLU
  ↓
Linear
  ↓
1 Raw Logit
  ↓
BCEWithLogitsLoss
```

The complete training pipeline is:

```text
Data
 ↓
ANN
 ↓
Raw logits
 ↓
Loss
 ↓
Backpropagation
 ↓
Gradients
 ↓
Optimizer
 ↓
Parameter update
```

---

# 2. Why Use an ANN Instead of a Perceptron?

### Classical perceptron

```text
Input
 ↓
Linear
 ↓
Step
 ↓
0 / 1
```

It can only create a **linear decision boundary**.

### Binary ANN

```text
Input
 ↓
Linear
 ↓
ReLU
 ↓
Linear
 ↓
Logit
```

The nonlinear activation allows the network to learn **nonlinear patterns**.

This is the key transition:

$$
\text{Perceptron}
\rightarrow
\text{Hidden layer}
\rightarrow
\text{Nonlinearity}
\rightarrow
\text{Modern ANN}
$$

---

# 3. Example Dataset

A simple synthetic dataset can be generated with:

```python
import torch

torch.manual_seed(42)

X = torch.randn(500, 2)

y = ((X[:, 0]**2 + X[:, 1]**2) > 1.0).float()
```

Shapes:

```text
X → (500, 2)
y → (500,)
```

Each sample has **2 features**.

The classification rule is based on:

$$
x_1^2+x_2^2>1
$$

This produces a nonlinear classification problem.

---

# 4. Train/Test Split 🔥🔥🔥

The model should not be evaluated only on the data it trained on.

Example:

```python
train_size = 400

X_train = X[:train_size]
y_train = y[:train_size]

X_test = X[train_size:]
y_test = y[train_size:]
```

Then:

```python
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train,
    y_train
)

test_dataset = TensorDataset(
    X_test,
    y_test
)
```

DataLoaders:

```python
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)
```

### Remember

```text
Training → shuffle=True
Testing  → shuffle=False
```

---

# 5. ANN Architecture 🔥🔥🔥

```python
import torch.nn as nn

class BinaryANN(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)
```

Architecture:

```text
(2 features)
     ↓
Linear(2 → 16)
     ↓
(16 features)
     ↓
ReLU
     ↓
Linear(16 → 1)
     ↓
(1 logit)
```

---

# 6. Why Does the Output Have One Neuron?

For binary classification, we can represent the prediction using **one raw logit**.

The model produces:

$$
z\in(-\infty,\infty)
$$

Examples:

```text
-4.2
-1.1
 0.3
 2.8
 7.1
```

This is **not yet a probability**.

It is the model's raw score, called a **logit**.

---

# 7. Logit → Probability

The sigmoid function converts a logit into a probability:

$$
p=\sigma(z)=\frac{1}{1+e^{-z}}
$$

Examples:

| Logit | Approx. probability |
| ----: | ------------------: |
|    -3 |                0.05 |
|    -1 |                0.27 |
|     0 |                0.50 |
|     1 |                0.73 |
|     3 |                0.95 |

Therefore:

```text
Negative logit → probability < 0.5
Positive logit → probability > 0.5
```

---

# 8. `BCEWithLogitsLoss` 🔥🔥🔥

For binary classification, use:

```python
loss_fn = nn.BCEWithLogitsLoss()
```

The important point:

> **The model should output raw logits.**

Don't normally put a `Sigmoid()` layer before `BCEWithLogitsLoss`.

### Correct

```text
Model
 ↓
Raw logits
 ↓
BCEWithLogitsLoss
```

### Avoid this normal training setup

```text
Model
 ↓
Sigmoid
 ↓
BCEWithLogitsLoss
```

`BCEWithLogitsLoss` combines the sigmoid operation with binary cross-entropy in a numerically stable implementation.

---

# 9. Why Sigmoid Is Used During Inference

During training:

```text
Logits
 ↓
BCEWithLogitsLoss
```

During inference, if we want probabilities:

```python
probabilities = torch.sigmoid(logits)
```

So:

```text
Raw logit
   ↓
Sigmoid
   ↓
Probability
```

For example:

$$
z=2
$$

gives approximately:

$$
\sigma(2)\approx0.88
$$

---

# 10. Probability → Class

For the standard 0.5 threshold:

$$
\hat y=
\begin{cases}
1 & p\geq0.5\\
0 & p<0.5
\end{cases}
$$

PyTorch:

```python
predictions = (
    probabilities >= 0.5
).float()
```

Example:

```text
0.91 → 1
0.72 → 1
0.48 → 0
0.12 → 0
```

---

# 11. Loss + Optimizer

```python
loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
```

Remember:

```text
Loss
 ↓
Measures prediction error

Optimizer
 ↓
Uses gradients to update parameters
```

---

# 12. Training Loop 🔥🔥🔥

```python
for epoch in range(num_epochs):

    model.train()

    for X_batch, y_batch in train_loader:

        # Forward
        logits = model(X_batch)

        # Loss
        loss = loss_fn(
            logits.squeeze(),
            y_batch
        )

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()
```

### The fundamental cycle

```text
Forward
   ↓
Loss
   ↓
zero_grad()
   ↓
backward()
   ↓
Gradients
   ↓
optimizer.step()
   ↓
Updated parameters
```

---

# 13. Understanding Tensor Shapes 🔥🔥🔥

Suppose:

```text
Batch size = 32
```

Input:

```text
X
(32, 2)
```

After:

```python
nn.Linear(2, 16)
```

we get:

```text
(32, 16)
```

After ReLU:

```text
(32, 16)
```

After:

```python
nn.Linear(16, 1)
```

we get:

```text
(32, 1)
```

Targets are:

```text
(32,)
```

Therefore:

```python
logits.squeeze()
```

converts:

```text
(32, 1)
```

to:

```text
(32,)
```

so it matches the target shape.

---

# 14. Why `.squeeze()`?

If:

```python
logits.shape
```

is:

```text
torch.Size([32, 1])
```

then:

```python
logits.squeeze().shape
```

becomes:

```text
torch.Size([32])
```

So:

```python
loss_fn(
    logits.squeeze(),
    y_batch
)
```

has matching shapes.

### Important

Don't blindly use `squeeze()` everywhere. Understand **which dimension you're removing** and make sure the resulting shape is what your loss expects.

---

# 15. Evaluation 🔥🔥🔥

After training:

```python
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        logits = model(X_batch)

        probabilities = torch.sigmoid(
            logits.squeeze()
        )

        predictions = (
            probabilities >= 0.5
        ).float()

        correct += (
            predictions == y_batch
        ).sum().item()

        total += y_batch.size(0)

accuracy = correct / total

print("Accuracy:", accuracy)
```

Important:

```python
model.eval()
```

changes model behavior for layers such as Dropout and BatchNorm.

```python
with torch.no_grad():
```

disables gradient tracking.

These are different concepts.

---

# 16. Complete Binary ANN Flow

```text
                 INPUT
                   │
                   ▼
              (Batch, 2)
                   │
                   ▼
             Linear 2→16
                   │
                   ▼
                 ReLU
                   │
                   ▼
             Linear 16→1
                   │
                   ▼
              RAW LOGIT
                   │
          ┌────────┴────────┐
          │                 │
       Training          Inference
          │                 │
          ▼                 ▼
BCEWithLogitsLoss         Sigmoid
          │                 │
          ▼                 ▼
      Backward          Probability
          │                 │
          ▼                 ▼
     Optimizer          Threshold
          │                 │
          ▼                 ▼
    Update params        0 / 1
```

---

# 17. Complete Implementation

```python
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# -------------------------
# Data
# -------------------------

torch.manual_seed(42)

X = torch.randn(500, 2)

y = (
    (X[:, 0]**2 + X[:, 1]**2) > 1.0
).float()

# Train/test split
train_size = 400

X_train = X[:train_size]
y_train = y[:train_size]

X_test = X[train_size:]
y_test = y[train_size:]

# Datasets
train_dataset = TensorDataset(
    X_train,
    y_train
)

test_dataset = TensorDataset(
    X_test,
    y_test
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# -------------------------
# Model
# -------------------------

class BinaryANN(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

model = BinaryANN()

# -------------------------
# Loss + Optimizer
# -------------------------

loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# -------------------------
# Training
# -------------------------

for epoch in range(100):

    model.train()

    for X_batch, y_batch in train_loader:

        logits = model(X_batch)

        loss = loss_fn(
            logits.squeeze(),
            y_batch
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

# -------------------------
# Evaluation
# -------------------------

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        logits = model(X_batch)

        probabilities = torch.sigmoid(
            logits.squeeze()
        )

        predictions = (
            probabilities >= 0.5
        ).float()

        correct += (
            predictions == y_batch
        ).sum().item()

        total += y_batch.size(0)

accuracy = correct / total

print("Test Accuracy:", accuracy)
```

---

# 18. Important Comparison

| Classical Perceptron     | Binary ANN                     |
| ------------------------ | ------------------------------ |
| One neuron               | Multiple layers                |
| Step activation          | ReLU + output logit            |
| 0/1 output directly      | Raw logit during training      |
| Perceptron update        | Backpropagation                |
| Linear decision boundary | Can learn nonlinear boundaries |
| Simple binary problems   | More complex patterns          |

---

# 19. Common Mistakes ⚠️

### ❌ Mistake 1

```python
nn.Sigmoid()
```

followed by:

```python
nn.BCEWithLogitsLoss()
```

### ✅ Correct

```text
Model → raw logits → BCEWithLogitsLoss
```

---

### ❌ Mistake 2

Using `CrossEntropyLoss` for a standard one-output binary setup without understanding the output/target formulation.

For our current setup:

```text
1 output logit
+
BCEWithLogitsLoss
```

---

### ❌ Mistake 3

Forgetting to convert logits to probabilities during inference.

```python
probabilities = torch.sigmoid(logits)
```

---

### ❌ Mistake 4

Forgetting:

```python
model.eval()
```

during evaluation.

---

### ❌ Mistake 5

Forgetting:

```python
with torch.no_grad():
```

during ordinary inference/evaluation.

---

### ❌ Mistake 6

Ignoring tensor shapes.

Typical setup:

```text
logits → (B, 1)
target → (B,)
```

so we use:

```python
logits.squeeze()
```

---

# 20. 🔬 Important Experiments

### Experiment 1 — Hidden-layer width

Compare:

```python
nn.Linear(2, 2)
```

vs

```python
nn.Linear(2, 16)
```

vs

```python
nn.Linear(2, 32)
```

Observe how model capacity changes.

---

### Experiment 2 — Learning rate

Try:

```text
0.1
0.01
0.001
0.0001
```

Observe convergence and stability.

---

### Experiment 3 — Optimizer

Compare:

```text
SGD
Momentum SGD
Adam
AdamW
```

Connect the observations to Module 12.

---

# 🎯 Must Know

## 🔥🔥🔥

You should be able to explain:

### Why one output neuron?

Binary classification can be represented by one logit.

### Why raw logits?

`BCEWithLogitsLoss` incorporates sigmoid in a numerically stable way.

### Why sigmoid during inference?

To convert the raw logit into a probability.

### Why threshold?

To convert probability into a binary class.

$$
p\geq0.5\Rightarrow1
$$

### Training

```text
Input
 ↓
ANN
 ↓
Logit
 ↓
BCEWithLogitsLoss
 ↓
Backward
 ↓
Optimizer
 ↓
Update
```

### Inference

```text
Input
 ↓
ANN
 ↓
Logit
 ↓
Sigmoid
 ↓
Probability
 ↓
Threshold
 ↓
Class
```

---

# 🧠 Final Cheat Sheet

### Model

```text
Input
 ↓
Linear
 ↓
ReLU
 ↓
Linear
 ↓
1 logit
```

### Training

```python
logits = model(X)
loss = loss_fn(logits.squeeze(), y)

optimizer.zero_grad()
loss.backward()
optimizer.step()
```

### Inference

```python
model.eval()

with torch.no_grad():
    logits = model(X)
    probability = torch.sigmoid(logits)
    prediction = (probability >= 0.5).float()
```

### 🔑 Memory sentence

> **Binary ANN training uses raw logits + `BCEWithLogitsLoss`; during inference, use sigmoid to obtain probabilities and a threshold to obtain classes.**

---
